# BigBasket Category Performance Diagnostic
Part 4: Python analysis and cross-validation.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\norders = pd.read_csv('orders_raw.csv')\nproducts = pd.read_csv('products.csv')\norders.head(), products.head()

In [ ]:
orders = orders.drop_duplicates(subset='order_id', keep='first').copy()\norders['city'] = orders.get('city', pd.Series(index=orders.index)).fillna('') if 'city' in orders else ''\nproducts['category'] = products['category'].astype(str).str.strip().str.title()\norders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')\norders['amount'] = pd.to_numeric(orders['amount'], errors='coerce')\norders.shape

In [ ]:
delivered = orders[orders['status'].eq('Delivered') & orders['amount'].notna()].copy()\nq1 = delivered['amount'].quantile(.25)\nq3 = delivered['amount'].quantile(.75)\niqr = q3-q1\nlower=q1-1.5*iqr; upper=q3+1.5*iqr\ndelivered['amount_capped']=delivered['amount'].clip(lower, upper)\nprint(q1,q3,lower,upper)

In [ ]:
delivered['month']=delivered['order_date'].dt.to_period('M').astype(str)\ncategory_summary=(delivered.merge(products,on='product_id')\n .groupby('category',as_index=False)['amount_capped'].sum()\n .sort_values('amount_capped',ascending=False))\nsupplier_summary=(delivered.merge(products,on='product_id')\n .groupby('supplier',as_index=False)['amount_capped'].sum()\n .sort_values('amount_capped',ascending=False))\nprint(category_summary)\nprint(supplier_summary.head())

In [ ]:
# Chart 1: monthly revenue\nm=delivered.merge(products,on='product_id').groupby('month')['amount_capped'].sum()\nplt.figure(figsize=(8,4)); m.plot(kind='line',marker='o'); plt.title('Monthly Delivered Revenue'); plt.xlabel('Month'); plt.ylabel('Revenue'); plt.tight_layout(); plt.show()

In [ ]:
# Chart 2: category revenue\ncategory_summary.set_index('category')['amount_capped'].plot(kind='bar',figsize=(8,4)); plt.title('Revenue by Category'); plt.ylabel('Revenue'); plt.xticks(rotation=35,ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Chart 3: supplier revenue\nsupplier_summary.head(8).set_index('supplier')['amount_capped'].plot(kind='bar',figsize=(8,4)); plt.title('Top Suppliers by Delivered Revenue'); plt.ylabel('Revenue'); plt.xticks(rotation=35,ha='right'); plt.tight_layout(); plt.show()

## What / Why / Next Step\n1. **What:** Household Essentials is the leading category in the delivered-order revenue summary. **Why:** its delivered sales total is the largest after joining orders to products. **Next:** review its product-level contribution and stock availability.\n2. **What:** Some categories fall below their assigned revenue targets. **Why:** their aggregated delivered revenue does not reach the target benchmark. **Next:** investigate order volume and average revenue by month.\n3. **What:** Supplier contribution is concentrated among the suppliers attached to the higher-revenue categories. **Why:** category-level product mix determines supplier exposure. **Next:** compare supplier performance with product availability and margins.